Uma análise feita sobre as rotas atuais demonstram que atualmente o formato de distribuição causará atrasos na maior parte das entregas. Esta análise pode ser vista pelo link: [1. Analysis - Rotas](https://github.com/gxlobato/DunderMifflin/blob/main/notebooks/1.%20Analysis%20-%20Rotas.ipynb)

Neste notebook pretendo explorar 3 cenários para a reestruturação

###Cenário A — Manter 4 armazéns, só mudar a regra (Centro-Oeste → Sudeste + Sul)
Reaproveita o que existe, mas talvez não resolva os piores casos (Centro-Oeste é geograficamente vasto).

###Cenário B — Realocar um armazém existente pra mais perto do Centro-Oeste
Por exemplo, trocar Recife (Nordeste) por Goiânia ou Brasília. Isso resolve o Centro-Oeste, mas potencialmente piora o Nordeste (que ficaria sem armazém próprio, teria que ser coberto por outro).

###Cenário C — Adicionar um 5º armazém no Centro-Oeste
Resolve o Centro-Oeste sem sacrificar nenhuma região existente, mas é mais armazém pra manter (custo/complexidade maior).

### Monta os candidatos e calcula a matriz completa

In [0]:
%pip install faker
dbutils.library.restartPython()

In [0]:
import sys
sys.path.append("/Workspace/Repos/Users/gabrielalbt@gmail.com/dunder-mifflin")  # ajusta pro caminho real do repo no Databricks
import os

import pandas as pd
from src.shared.municipios import buscar_lat_long
from src.logistica.distancias import calcula_distancias

DIR_RAW = "/Volumes/workspace/dundermiffin/raw"
DIR_SOURCE = "/Volumes/workspace/dundermiffin/source"
DIR_CACHE = "/Volumes/workspace/dundermiffin/cache"

api_key = dbutils.widgets.get('api_key')

def carregar_parquet(caminho):
    """Carrega um DataFrame salvo em Parquet, ou None se o arquivo não existir."""
    if not os.path.exists(caminho):
        return None
    return pd.read_parquet(caminho)

# carrega tudo que já foi gerado pelo pipeline principal, sem recalcular nada
df_municipios = carregar_parquet(f"{DIR_RAW}/municipios.parquet")
df_lat_long = carregar_parquet(f"{DIR_RAW}/lat_long.parquet")
df_armazem = carregar_parquet(f"{DIR_SOURCE}/armazens.parquet")
df_clientes = carregar_parquet(f"{DIR_SOURCE}/clientes.parquet")

for nome, df in [
    ('municipios', df_municipios), ('lat_long', df_lat_long),
    ('armazem', df_armazem), ('clientes', df_clientes)
]:
    if df is None:
        raise FileNotFoundError(f"{nome}.parquet não encontrado — rode o notebook principal primeiro.")

In [0]:
# Candidatos a armazém no Centro-Oeste, para testar realocação (Cenário B)
# ou adição de um 5º armazém (Cenário C)
candidatos_extra = [
    {'cidade': 'Goiânia', 'codigo': 'A5'},
    {'cidade': 'Brasília', 'codigo': 'A6'},
]

def montar_candidatos(df_municipios, df_lat_long, candidatos, id_inicial):
    """Monta armazéns candidatos extras, no mesmo formato de montar_armazens()."""
    linhas = []
    for i, c in enumerate(candidatos):
        linha = df_municipios[df_municipios['nome_cidade'] == c['cidade']].head(1)
        if linha.empty:
            raise ValueError(f"Cidade não encontrada: {c['cidade']}")

        codigo_ibge = linha['codigo_ibge'].values[0]
        uf = linha['uf'].values[0]
        regiao = linha['regiao'].values[0]
        latitude, longitude = buscar_lat_long(df_lat_long, codigo_ibge)

        linhas.append({
            'id': id_inicial + i,
            'codigo': c['codigo'],
            'cidade': c['cidade'],
            'uf': uf,
            'regiao': regiao,
            'latitude': latitude,
            'longitude': longitude,
        })
    return pd.DataFrame(linhas)

# junta os 4 armazéns reais + os candidatos, com ids sequenciais (5, 6...)
df_candidatos = montar_candidatos(df_municipios, df_lat_long, candidatos_extra, id_inicial=df_armazem['id'].max() + 1)
df_armazem_todos = pd.concat([df_armazem, df_candidatos], ignore_index=True)

print(df_armazem_todos[['id', 'codigo', 'cidade', 'regiao']])